## Bonus Assignment 4 - BERT Pre-training

###  **Objective**

Implement the full BERT architecture and its two pre-training objectives: **Masked Language Modeling (MLM)** and **Next Sentence Prediction (NSP)**. You will build:

1.  A `torch.utils.data.Dataset` to handle the complex 80/10/10 masking (MLM) and sentence pairing (NSP).
2.  The `BertModel` by stacking your `EncoderLayer` and adding token-type (segment) embeddings.
3.  The final `BertForPretraining` model, which includes the MLM and NSP prediction heads and calculates the combined loss.

###  **Assignment Structure**

This assignment will contain:

1.  **Setup & Prerequisite Code:** Imports, configurations, a simple corpus, a basic tokenizer, and the *completed* solutions for all modules from Weeks 1 & 2.
2.  **Part 1: The Pre-training Data Pipeline:** A `Dataset` class to create MLM-masked tokens and NSP sentence pairs.
3.  **Part 2: The `BertModel` (Encoder Stack):** Assembling the full BERT body.
4.  **Part 3: The `BertForPretraining` (Heads & Loss):** Adding the final layers to compute the MLM and NSP losses.

In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import random
import collections
from torch.utils.data import Dataset, DataLoader

# --- Reproducibility ---
torch.manual_seed(42)
random.seed(42)

# --- Configuration ---
# Small config for verifiable testing
D_MODEL = 64
NUM_HEADS = 4
NUM_LAYERS = 2  # A 2-layer BERT
D_FF = 128
DROPOUT = 0.1
BATCH_SIZE = 4
SEQ_LEN = 20 # Max sequence length

# --- Sample Corpus ---
corpus = [
    "this is the first sentence.",
    "this is the second one.",
    "here is a third sentence.",
    "and a fourth one follows.",
    "bert is a powerful model.",
    "it was trained on much data.",
    "masked language modeling is key.",
    "so is next sentence prediction."
]

# --- Simple Tokenizer ---
# (Builds a simple word-level vocab)
def get_tokenizer(corpus):
    words = set(word for sent in corpus for word in sent.strip('.').split())
    vocab = {'[PAD]': 0, '[CLS]': 1, '[SEP]': 2, '[MASK]': 3}
    vocab.update({word: i+4 for i, word in enumerate(words)})
    return vocab

tokenizer = get_tokenizer(corpus)
VOCAB_SIZE = len(tokenizer)
print(f"Using D_MODEL={D_MODEL}, VOCAB_SIZE={VOCAB_SIZE}, SEQ_LEN={SEQ_LEN}")

# --- Prerequisite Code (Solutions from W1 & W2) ---
print("Loading prerequisite code...")
def scaled_dot_product_attention(q, k, v, mask=None):
    d_k=k.size(-1); scores=torch.matmul(q,k.transpose(-2,-1))/math.sqrt(d_k)
    if mask is not None: scores=scores.masked_fill(mask==0,-1e9)
    p_attn=torch.softmax(scores,dim=-1); return torch.matmul(p_attn,v),p_attn
class MultiHeadAttention(nn.Module):
    def __init__(self, d, n): super().__init__(); self.d,self.n,self.h=d,n,d//n; self.wq,self.wk,self.wv,self.wo=(nn.Linear(d,d) for _ in range(4))
    def forward(self, q, k, v, mask=None):
        b=q.size(0); Q,K,V=(l(x).view(b,-1,self.n,self.h).transpose(1,2) for l,x in [(self.wq,q),(self.wk,k),(self.wv,v)])
        x,_=scaled_dot_product_attention(Q,K,V,mask); x=x.transpose(1,2).contiguous().view(b,-1,self.d); return self.wo(x)
class PositionWiseFeedForward(nn.Module):
    def __init__(self, d, d_ff): super().__init__(); self.l1,self.r,self.l2=nn.Linear(d,d_ff),nn.ReLU(),nn.Linear(d_ff,d)
    def forward(self, x): return self.l2(self.r(self.l1(x)))
class EncoderLayer(nn.Module):
    def __init__(self, d, n, d_ff, dr=0.1):
        super().__init__(); self.mha=MultiHeadAttention(d,n); self.ffn=PositionWiseFeedForward(d,d_ff)
        self.n1,self.n2=(nn.LayerNorm(d) for _ in range(2)); self.d1,self.d2=(nn.Dropout(dr) for _ in range(2))
    def forward(self, x, mask=None):
        x = self.n1(x + self.d1(self.mha(x,x,x,mask))); return self.n2(x + self.d2(self.ffn(x)))
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__(); pe=torch.zeros(max_len,d_model); pos=torch.arange(0,max_len,dtype=torch.float).unsqueeze(1)
        div=torch.exp(torch.arange(0,d_model,2).float()*(-math.log(10000.0)/d_model)); pe[:,0::2]=torch.sin(pos*div); pe[:,1::2]=torch.cos(pos*div)
        self.register_buffer('pe',pe.unsqueeze(0))
    def forward(self, x): return x + self.pe[:,:x.size(1),:]
print("Prerequisite code loaded successfully.")

Using D_MODEL=64, VOCAB_SIZE=33, SEQ_LEN=20
Loading prerequisite code...
Prerequisite code loaded successfully.


-----

### **Part 1: The Pre-training Data Pipeline**

**Instructions:** Create a `Dataset` class. The `__init__` will create sentence pairs. The `__getitem__` will be responsible for tokenization, padding, and applying the MLM 80/10/10 masking rule.

In [13]:
print("\n--- Part 1: The Pre-training Data Pipeline ---")

class BertPretrainingDataset(Dataset):
    def __init__(self, corpus, tokenizer, seq_len):
        self.tokenizer = tokenizer
        self.seq_len = seq_len
        self.corpus_lines = corpus
        self.vocab_items = list(tokenizer.keys())

        # --- Create Sentence Pairs ---
        self.items = []
        for i in range(len(self.corpus_lines) - 1):
            # 1. Add "IsNext" pair
            self.items.append((self.corpus_lines[i], self.corpus_lines[i+1], 1)) # 1 = IsNext

            # 2. Add "NotNext" pair
            rand_idx = i
            while rand_idx == i or rand_idx == i+1:
                 rand_idx = random.randint(0, len(self.corpus_lines) - 1)
            self.items.append((self.corpus_lines[i], self.corpus_lines[rand_idx], 0)) # 0 = NotNext

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        # 1. Get the sentence pair and NSP label
        sent_a, sent_b, nsp_label = self.items[idx]

        # 2. Tokenize and add special tokens
        tokens_a = [self.tokenizer.get(w, 0) for w in sent_a.strip('.').split()]
        tokens_b = [self.tokenizer.get(w, 0) for w in sent_b.strip('.').split()]

        # Combine: [CLS] + sent_a + [SEP] + sent_b + [SEP]
        input_ids = [self.tokenizer['[CLS]']] + tokens_a + [self.tokenizer['[SEP]']] + tokens_b + [self.tokenizer['[SEP]']]

        # 3. Create Token Type IDs (Segment IDs)
        # 0 for [CLS] and sent_a, 1 for sent_b
        segment_ids = [0] * (len(tokens_a) + 2) + [1] * (len(tokens_b) + 1)

        # 4. Apply MLM (This is the complex part)
        # We will set a random seed *inside* __getitem__ for reproducibility in grading
        # In real training, you would NOT do this.
        torch.manual_seed(42 + idx)
        random.seed(42 + idx)

        # `input_ids_mlm` will have masks, `mlm_labels` will have the original tokens
        input_ids_mlm, mlm_labels = self._create_mlm_data(input_ids)

        # 5. Pad all sequences to `self.seq_len`
        pad_len = self.seq_len - len(input_ids_mlm)
        pad_token = self.tokenizer['[PAD]']

        input_ids_mlm = input_ids_mlm + [pad_token] * pad_len
        segment_ids = segment_ids + [pad_token] * pad_len
        mlm_labels = mlm_labels + [-100] * pad_len # -100 is the ignore_index for CrossEntropyLoss

        return (torch.tensor(input_ids_mlm[:self.seq_len]),
                torch.tensor(segment_ids[:self.seq_len]),
                torch.tensor(mlm_labels[:self.seq_len]),
                torch.tensor(nsp_label))

    def _create_mlm_data(self, input_ids):
        """Applies the 80/10/10 masking rule to a list of token IDs."""
        input_ids_mlm = list(input_ids)
        # We use -100 to ignore non-masked tokens in the loss function
        mlm_labels = [-100] * len(input_ids)

        # Find all token indices that are *not* special tokens
        candidate_indices = [
            i for i, token_id in enumerate(input_ids)
            if token_id not in [self.tokenizer['[CLS]'], self.tokenizer['[SEP]'], self.tokenizer['[PAD]']]
        ]

        # Determine how many tokens to mask (15% of candidates)
        num_to_mask = max(1, int(len(candidate_indices) * 0.15))
        random.shuffle(candidate_indices)
        indices_to_mask = candidate_indices[:num_to_mask]

        for i in indices_to_mask:
            original_token_id = input_ids[i]
            # Store the original token as the label
            mlm_labels[i] = original_token_id

            # 80% of the time: Replace with [MASK]
            if random.random() < 0.8:
                input_ids_mlm[i] = self.tokenizer['[MASK]']
            # 10% of the time: Replace with a random token
            elif random.random() < 0.5: # (This is 0.5 of the remaining 0.2, so 10% total)
                random_token_id = random.choice(list(self.tokenizer.values()))
                input_ids_mlm[i] = random_token_id
            # 10% of the time: Keep the original token
            else:
                input_ids_mlm[i] = original_token_id

        return input_ids_mlm, mlm_labels

# --- Verification for Part 1 ---
print("--- Running Verification for Part 1 ---")
try:
    torch.manual_seed(42)
    random.seed(42)
    dataset = BertPretrainingDataset(corpus, tokenizer, SEQ_LEN)

    # Fetch the first item
    item = dataset[0] # ('this is the first sentence.', 'this is the second one.', 1)
    input_ids, segment_ids, mlm_labels, nsp_label = item

    assert input_ids.shape == (SEQ_LEN,)
    assert segment_ids.shape == (SEQ_LEN,)
    assert mlm_labels.shape == (SEQ_LEN,)
    print(" (1a) All output shapes are correct.")

    assert nsp_label == 1
    print(" (1b) NSP label is correct for item 0.")

    # Check that masking happened
    assert (mlm_labels != -100).sum().item() > 0
    print(" (1c) MLM labels were created.")

    # Check that input was masked
    assert torch.all(input_ids[mlm_labels != -100] != mlm_labels[mlm_labels != -100])
    print(" (1d) 80/10 masking rule was applied (at least the 80% or 10% part).")

    print(" Part 1 seems correct!")
except Exception as e:
    print(f" Part 1 Failed: {e}")


--- Part 1: The Pre-training Data Pipeline ---
--- Running Verification for Part 1 ---
 (1a) All output shapes are correct.
 (1b) NSP label is correct for item 0.
 (1c) MLM labels were created.
 (1d) 80/10 masking rule was applied (at least the 80% or 10% part).
 Part 1 seems correct!


#### **Questions for Part 1**

**Q1: Conceptual Check**
What is the purpose of the `mlm_labels` tensor containing `-100` for all non-masked tokens?

  * (A) It saves memory by not storing labels for every token.
  * (B) `-100` is the `ignore_index` for `nn.CrossEntropyLoss`, so the loss is only calculated on the tokens that were actually masked.
  * (C) It signals to the model which tokens to pay attention to.
  * (D) It's an arbitrary padding value that could also be 0.


**Q2: Value Check**
Using the `dataset` created in the verification block, fetch `item[1]`. (This is the "NotNext" pair: `"this is the first sentence."`, `"bert is a powerful model."`). What is the **sum** of the `segment_ids` tensor?

In [14]:
# Code for Q2
torch.manual_seed(42); random.seed(42)
item = dataset[1]
segment_ids = item[1]
print(f"Q2 Segment IDs Sum: {segment_ids.sum().item()}")

Q2 Segment IDs Sum: 6




**Q3: Value Check**
Using the same `dataset_q2[1]`, what is the **sum** of the `mlm_labels` tensor? (Remember, most values are -100, so this will be a negative number).

In [15]:
# Code for Q3
torch.manual_seed(42); random.seed(42)
print(f"Q3 MLM Labels Sum: {item[2].sum().item()}")

Q3 MLM Labels Sum: -1874




-----

### **Part 2: The `BertModel` (Encoder Stack)**

**Instructions:** Now, assemble the main `BertModel`. This module will be responsible for applying all the embeddings (Token, Segment, Position) and then passing the result through a stack of `EncoderLayer` modules (which you already built).

In [16]:
print("\n--- Part 2: The BertModel (Encoder Stack) ---")

class BertModel(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, num_layers, d_ff, dropout, seq_len):
        super(BertModel, self).__init__()

        # 1. Token Embeddings (nn.Embedding)
        self.word_embedding = nn.Embedding(vocab_size, d_model)

        # 2. Segment Embeddings (nn.Embedding)
        # (Size 2: for sentence A and sentence B)
        self.segment_embedding = nn.Embedding(2, d_model)

        # 3. Positional Encodings (Use the provided W2 class)
        self.pos_encoding = PositionalEncoding(d_model, seq_len)

        # 4. Stack of N EncoderLayers
        self.layers = nn.ModuleList(
            [EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)]
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, input_ids, segment_ids, padding_mask=None):
        # input_ids: (batch_size, seq_len)
        # segment_ids: (batch_size, seq_len)

        # 1. Get word embeddings
        word_embeds = self.word_embedding(input_ids)

        # 2. Get segment embeddings
        seg_embeds = self.segment_embedding(segment_ids)

        # 3. Add all three embeddings together
        x = word_embeds + seg_embeds

        # 4. Add positional encodings
        x = self.pos_encoding(x)
        x = self.dropout(x)

        # 4. Pass through the N encoder layers
        for layer in self.layers:
            x = layer(x, padding_mask)

        return x

# --- Verification for Part 2 ---
print("--- Running Verification for Part 2 ---")
try:
    torch.manual_seed(42)
    # Use the global config variables
    bert_model = BertModel(VOCAB_SIZE, D_MODEL, NUM_HEADS, NUM_LAYERS, D_FF, DROPOUT, SEQ_LEN)

    # Create dummy inputs
    test_input_ids = torch.randint(0, VOCAB_SIZE, (BATCH_SIZE, SEQ_LEN))
    test_seg_ids = torch.randint(0, 2, (BATCH_SIZE, SEQ_LEN))

    output = bert_model(test_input_ids, test_seg_ids)

    assert output.shape == (BATCH_SIZE, SEQ_LEN, D_MODEL)
    print(" (2a) Final output shape is correct.")
    print(" Part 2 seems correct!")
except Exception as e:
    print(f" Part 2 Failed: {e}")


--- Part 2: The BertModel (Encoder Stack) ---
--- Running Verification for Part 2 ---
 (2a) Final output shape is correct.
 Part 2 seems correct!


#### **Questions for Part 2**

**Q4: Conceptual Check**
In this `BertModel`, how does the model differentiate between the first sentence (`sent_a`) and the second sentence (`sent_b`) in a pair?

  * (A) By using the `PositionalEncoding`.
  * (B) By using the `self.segment_embedding` (token type IDs).
  * (C) It doesn't; only the `[SEP]` token separates them.
  * (D) By using the `padding_mask`.



**Q5: Value Check**
Run the code block below to initialize the `BertModel` (with `seed=42`) and pass a fixed input through it. What is the **mean** of the resulting `output` tensor?

In [17]:
# Code for Q5
torch.manual_seed(42)
bert_model_q5 = BertModel(VOCAB_SIZE, D_MODEL, NUM_HEADS, NUM_LAYERS, D_FF, DROPOUT, SEQ_LEN)
# Create a fixed input tensor
test_input_ids_q5 = (torch.arange(BATCH_SIZE * SEQ_LEN) % VOCAB_SIZE).view(BATCH_SIZE, SEQ_LEN)
test_seg_ids_q5 = (torch.arange(BATCH_SIZE * SEQ_LEN) % 2).view(BATCH_SIZE, SEQ_LEN)

output_q5 = bert_model_q5(test_input_ids_q5, test_seg_ids_q5)
print(f"Q5 Output Mean: {output_q5.mean().item()}")

Q5 Output Mean: -7.450580596923828e-09




-----

### **Part 3: `BertForPretraining` (Heads & Loss)**

**Instructions:** This is the final model. It will contain the `BertModel` from Part 2 and add the two "heads" required for pre-training. The `forward` pass will compute and return the **total combined loss**.

In [18]:
print("\n--- Part 3: BertForPretraining (Heads & Loss) ---")

class BertForPretraining(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, num_layers, d_ff, dropout, seq_len):
        super(BertForPretraining, self).__init__()
        # 1. The main BERT model
        self.bert = BertModel(vocab_size, d_model, num_heads, num_layers, d_ff, dropout, seq_len)

        # 2. The Next Sentence Prediction (NSP) head
        # A simple linear classifier on the [CLS] token output
        self.nsp_head = nn.Linear(d_model, 2)

        # 3. The Masked Language Model (MLM) head
        # This is often a small 2-layer MLP
        self.mlm_head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ReLU(),
            nn.Linear(d_model, vocab_size)
        )

        self.loss_fct = nn.CrossEntropyLoss() # This handles ignore_index=-100

    def forward(self, input_ids, segment_ids, mlm_labels, nsp_label):
        # 1. Get the final hidden states from the BERT model
        # `encoder_output` shape: (batch_size, seq_len, d_model)
        encoder_output = self.bert(input_ids, segment_ids)

        # 2. --- NSP ---
        # 2a. Get the output for the [CLS] token (it's at index 0)
        cls_token_output = encoder_output[:, 0]
        # 2b. Get the NSP logits
        nsp_logits = self.nsp_head(cls_token_output)

        # 3. --- MLM ---
        # 3a. Get the logits for *all* tokens
        mlm_logits = self.mlm_head(encoder_output)

        # 4. --- Calculate Losses ---
        # 4a. MLM Loss
        # We must reshape logits to (Batch*Seq, Vocab) and labels to (Batch*Seq)
        mlm_loss = self.loss_fct(
            mlm_logits.view(-1, VOCAB_SIZE),
            mlm_labels.view(-1)
        )

        # 4b. NSP Loss
        # Reshape logits to (Batch, 2) and labels to (Batch)
        nsp_loss = self.loss_fct(
            nsp_logits.view(-1, 2),
            nsp_label.view(-1)
        )

        # 5. Total Loss
        total_loss = mlm_loss + nsp_loss

        return total_loss, nsp_logits, mlm_logits

# --- Verification for Part 3 ---
print("--- Running Verification for Part 3 ---")
try:
    torch.manual_seed(42)
    random.seed(42)

    # 1. Create the final model
    model = BertForPretraining(VOCAB_SIZE, D_MODEL, NUM_HEADS, NUM_LAYERS, D_FF, DROPOUT, SEQ_LEN)

    # 2. Create the dataset and dataloader
    dataset_v3 = BertPretrainingDataset(corpus, tokenizer, SEQ_LEN)
    loader_v3 = DataLoader(dataset_v3, batch_size=BATCH_SIZE, shuffle=False) # No shuffle for test

    # 3. Get the first batch
    batch = next(iter(loader_v3))
    input_ids, segment_ids, mlm_labels, nsp_label = batch

    # 4. Run the model
    total_loss, nsp_logits, mlm_logits = model(input_ids, segment_ids, mlm_labels, nsp_label)

    assert total_loss.dim() == 0 and total_loss > 0
    print(" (3a) Total loss is a positive scalar.")
    assert nsp_logits.shape == (BATCH_SIZE, 2)
    print("(3b) NSP logits shape is correct.")
    assert mlm_logits.shape == (BATCH_SIZE, SEQ_LEN, VOCAB_SIZE)
    print("(3c) MLM logits shape is correct.")
    print("Part 3 seems correct!")
except Exception as e:
    print(f"Part 3 Failed: {e}")


--- Part 3: BertForPretraining (Heads & Loss) ---
--- Running Verification for Part 3 ---
 (3a) Total loss is a positive scalar.
(3b) NSP logits shape is correct.
(3c) MLM logits shape is correct.
Part 3 seems correct!


#### **Questions for Part 3**

**Q6: Conceptual Check**
For the NSP loss, we only use the encoder's output corresponding to the `[CLS]` token. Why?

  * (A) It's the only token that has a segment ID of 0.
  * (B) It's computationally cheaper than using all tokens.
  * (C) By convention, the `[CLS]` token's final hidden state is designed to be an aggregate representation of the entire sequence pair, suitable for classification tasks.
  * (D) Because the `[CLS]` token is never masked by MLM.



**Q7: Value Check**
Using the exact setup from the "Verification for Part 3" block (including `seed=42`), what is the `total_loss.item()` for the **first batch**?

In [19]:
# Code for Q7
torch.manual_seed(42)
random.seed(42)
model_q7 = BertForPretraining(VOCAB_SIZE, D_MODEL, NUM_HEADS, NUM_LAYERS, D_FF, DROPOUT, SEQ_LEN)
dataset_q7 = BertPretrainingDataset(corpus, tokenizer, SEQ_LEN)
loader_q7 = DataLoader(dataset_q7, batch_size=BATCH_SIZE, shuffle=False)
batch_q7 = next(iter(loader_q7))
input_ids_q7, segment_ids_q7, mlm_labels_q7, nsp_label_q7 = batch_q7
total_loss_q7, _, _ = model_q7(input_ids_q7, segment_ids_q7, mlm_labels_q7, nsp_label_q7)
print(f"Q7 Total Loss: {total_loss_q7.item()}")

Q7 Total Loss: 4.243647575378418



**Q8: Value Check**
Using the same `model_q7` and `batch_q7` from the previous question, what is the **mean** of the `nsp_logits` tensor?

In [20]:
# Code for Q8
torch.manual_seed(42)
random.seed(42)
model_q8 = BertForPretraining(VOCAB_SIZE, D_MODEL, NUM_HEADS, NUM_LAYERS, D_FF, DROPOUT, SEQ_LEN)
dataset_q8 = BertPretrainingDataset(corpus, tokenizer, SEQ_LEN)
loader_q8 = DataLoader(dataset_q8, batch_size=BATCH_SIZE, shuffle=False)
batch_q8 = next(iter(loader_q8))
input_ids_q8, segment_ids_q8, mlm_labels_q8, nsp_label_q8 = batch_q8
_, nsp_logits_q8, _ = model_q8(input_ids_q8, segment_ids_q8, mlm_labels_q8, nsp_label_q8)
print(f"Q8 NSP Logits Mean: {nsp_logits_q8.mean().item()}")

Q8 NSP Logits Mean: 0.48107656836509705
